In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import time

# Song Recommendation System - Two-Tower Neural Network

## Roadmap & Struktur Notebook

- Pendahuluan & Import Library
- Data Preparation
  - Load & Preprocess
  - Advanced Augmentation & Oversampling
  - Build Vocab & Song Mapping
  - Dataset & DataLoader
- Model Architecture
  - Konfigurasi Model
  - Pretrained Embedding (FastText)
  - MessageTower (GRU/Attention)
  - SongTower
- Training Pipeline
  - Fungsi Training (dengan Early Stopping, Label Smoothing, Scheduler)
  - Training Dua Model (GRU-only & GRU+Attention)
  - Ensemble
- Evaluasi & Analisis
  - Top-K Accuracy
  - Grid Search Hyperparameter
  - K-Fold Cross-Validation
  - Error Analysis
- (Opsional) Inference & Deployment

---

Setiap bagian utama akan diberi heading dan subheading yang jelas agar mudah dinavigasi.


In [ ]:
# 1. Load & Preprocess
train_df = pd.read_csv('all_songs_data_50k.csv')
song_df = pd.read_csv('songs.csv')
validation_df = pd.read_csv('validation_dataset.csv')

# Gabungkan train dan validation
full_df = pd.concat([train_df, validation_df], ignore_index=True)

# Data Preparation

Bagian ini memuat seluruh proses persiapan data, mulai dari load, cleaning, augmentasi, hingga pembuatan DataLoader.


In [ ]:
# --- Konfigurasi Model & Pelatihan ---
CONFIG = {
    # Arsitektur Model
    "text_embedding_dim": 192, # dikurangi agar tidak overfit
    "hidden_dim": 128, # dikurangi
    "n_layers": 2,

    # Regularisasi lebih kuat
    "dropout": 0.5, # lebih besar
    "weight_decay": 1e-3, # lebih besar

    # Konfigurasi Pelatihan
    "output_embedding_dim": 64, # dikurangi
    "batch_size": 128,
    "learning_rate": 0.00015, # lebih kecil
    "epochs": 25, # lebih sedikit
    
    # Lainnya
    "max_len": 50,
    "margin": 0.2,
    "min_song_freq": 5,
}

## Model Architecture & Configuration

Bagian ini berisi konfigurasi hyperparameter, opsi embedding, dan arsitektur model (MessageTower & SongTower).


In [ ]:
# --- 1. Persiapan Device (CPU/GPU) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}. Pelatihan akan menggunakan GPU.")
else:
    device = torch.device("cpu")
    print("GPU tidak ditemukan. Pelatihan akan menggunakan CPU.")

GPU terdeteksi: NVIDIA GeForce GTX 1650 Ti. Pelatihan akan menggunakan GPU.


In [ ]:
# --- 2. Simulasi dan Pra-pemrosesan Data ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_vocab(all_texts):
    word_counts = Counter(word for text in all_texts for word in text.split())
    vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common())}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

In [ ]:
# --- MessageTower dengan Pretrained Embedding & Attention ---
class MessageTower(nn.Module):
    def __init__(self, vocab_size, config, vocab=None):
        super().__init__()
        self.config = config
        if config.get("use_fasttext", False) and vocab is not None:
            # Load FastText embedding
            import gensim
            print("Memuat FastText embedding...")
            ft = gensim.models.KeyedVectors.load_word2vec_format(config["fasttext_path"], limit=100000)
            embedding_matrix = np.zeros((vocab_size, config["fasttext_dim"]))
            for word, idx in vocab.items():
                if word in ft:
                    embedding_matrix[idx] = ft[word]
                else:
                    embedding_matrix[idx] = np.random.normal(scale=0.6, size=(config["fasttext_dim"]))
            self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False, padding_idx=0)
            emb_dim = config["fasttext_dim"]
        else:
            self.embedding = nn.Embedding(vocab_size, config["text_embedding_dim"], padding_idx=0)
            emb_dim = config["text_embedding_dim"]
        self.gru = nn.GRU(
            emb_dim,
            config["hidden_dim"],
            num_layers=config["n_layers"],
            dropout=config["dropout"],
            batch_first=True
        )
        # Attention opsional
        self.use_attention = config.get("use_attention", False)
        if self.use_attention:
            self.attn = nn.Linear(config["hidden_dim"], 1)
        self.fc = nn.Linear(config["hidden_dim"], config["output_embedding_dim"])

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        if self.use_attention:
            # Attention over time steps
            attn_weights = torch.softmax(self.attn(output).squeeze(-1), dim=1).unsqueeze(-1)
            context = (output * attn_weights).sum(dim=1)
        else:
            context = hidden[-1]
        return F.normalize(self.fc(context), p=2, dim=1)


#### MessageTower dengan Pretrained Embedding & Attention

Kelas MessageTower berikut mendukung dua fitur opsional: (1) inisialisasi embedding dengan pretrained FastText, dan (2) penambahan attention layer untuk menangkap konteks kata lebih baik.


In [ ]:
# --- 4. Dataset dan DataLoader ---
class RecommendationDataset(Dataset):
    def __init__(self, messages, song_data, vocab, song_map, max_len):
        self.messages = messages
        self.song_data = song_data # Berisi song_id (bukan nama lagu)
        self.vocab = vocab
        self.song_map = song_map
        self.max_len = max_len

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        # Ambil pasangan (pesan, song_id) yang positif
        message = self.messages[idx]
        song_id = self.song_data[idx]  # song_id langsung
        
        # Vectorize pesan
        vectorized_msg = [self.vocab.get(word, self.vocab['<UNK>']) for word in message.split()]
        if len(vectorized_msg) < self.max_len:
            vectorized_msg += [self.vocab['<PAD>']] * (self.max_len - len(vectorized_msg))
        else:
            vectorized_msg = vectorized_msg[:self.max_len]
        
        # Dapatkan ID lagu dari map
        mapped_song_id = self.song_map[song_id]
        
        return torch.tensor(vectorized_msg, dtype=torch.long), torch.tensor(mapped_song_id, dtype=torch.long)

In [121]:
# --- 5. Fungsi Pelatihan ---
def train_recommendation_model(message_tower, song_tower, train_loader, val_loader, config, device, early_stopping_patience=7, save_path='best_model.pth'):
    message_tower.to(device)
    song_tower.to(device)
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"],
        weight_decay=config.get("weight_decay", 0.0)
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])

    def compute_accuracy_and_loss(loader, message_tower, song_tower, device, criterion):
        message_tower.eval()
        song_tower.eval()
        correct = 0
        total = 0
        total_loss = 0
        with torch.no_grad():
            for messages, song_ids in loader:
                messages, song_ids = messages.to(device), song_ids.to(device)
                msg_emb = message_tower(messages)
                all_song_ids = torch.arange(song_tower.embedding.num_embeddings).to(device)
                song_embs = song_tower(all_song_ids)
                sims = torch.matmul(msg_emb, song_embs.T)  # (batch, num_songs)
                preds = torch.argmax(sims, dim=1)
                correct += (preds == song_ids).sum().item()
                total += messages.size(0)
                # Loss: positif dan negatif
                pos_emb = song_embs[song_ids]
                loss_pos = criterion(msg_emb, pos_emb, torch.ones(messages.size(0)).to(device))
                neg_emb = torch.roll(pos_emb, shifts=1, dims=0)
                loss_neg = criterion(msg_emb, neg_emb, -torch.ones(messages.size(0)).to(device))
                total_loss += (loss_pos + loss_neg).item()
        avg_loss = total_loss / len(loader) if len(loader) > 0 else 0.0
        acc = correct / total if total > 0 else 0.0
        return acc, avg_loss

    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    best_gap = float('inf')
    best_general_model = None
    best_general_epoch = 0
    best_general_metrics = None
    for epoch in range(1, config["epochs"] + 1):
        start_time = time.time()
        message_tower.train()
        song_tower.train()
        total_loss = 0
        for messages, positive_songs in train_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            optimizer.zero_grad()
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            loss = loss_pos + loss_neg
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        train_acc, train_loss = compute_accuracy_and_loss(train_loader, message_tower, song_tower, device, criterion)
        val_acc, val_loss = compute_accuracy_and_loss(val_loader, message_tower, song_tower, device, criterion)
        end_time = time.time()
        print(f"Epoch: {epoch}/{config['epochs']} | "
              f"Waktu: {end_time - start_time:.2f}s | "
              f"Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}")
        # Early stopping logic (val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            # Simpan model terbaik val_loss
            torch.save({
                'message_tower_state_dict': message_tower.state_dict(),
                'song_tower_state_dict': song_tower.state_dict(),
                'vocab': vocab,
                'song_map': song_map
            }, save_path)
            print(f"  [Model terbaik val_loss disimpan di epoch {epoch}]")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping di epoch {epoch} (tidak ada perbaikan val_loss selama {early_stopping_patience} epoch)")
                break
        # Simpan model dengan gap train/val loss terkecil (model paling general)
        gap = abs(train_loss - val_loss)
        if gap < best_gap and train_loss < 2*val_loss and val_loss < 2*train_loss:  # syarat: tidak underfit/overfit ekstrim
            best_gap = gap
            best_general_epoch = epoch
            best_general_metrics = (train_loss, val_loss, train_acc, val_acc)
            best_general_model = {
                'message_tower_state_dict': message_tower.state_dict(),
                'song_tower_state_dict': song_tower.state_dict(),
                'vocab': vocab,
                'song_map': song_map
            }
            torch.save(best_general_model, 'best_general_model.pth')
            print(f"  [Model paling general (gap loss minimum) disimpan di epoch {epoch}]")
    print(f"Training selesai. Model terbaik val_loss pada epoch {best_epoch} dengan val_loss {best_val_loss:.4f}.")
    if best_general_metrics:
        print(f"Model paling general (gap loss minimum) pada epoch {best_general_epoch} | Train loss: {best_general_metrics[0]:.4f} | Val loss: {best_general_metrics[1]:.4f} | Train acc: {best_general_metrics[2]:.4f} | Val acc: {best_general_metrics[3]:.4f}")

## Training Pipeline

Bagian ini mencakup seluruh proses training, mulai dari fungsi pelatihan, training dua model, hingga ensemble.


In [122]:
# --- Evaluasi Top-K Accuracy ---
def top_k_accuracy(loader, message_tower, song_tower, device, k=5):
    message_tower.eval()
    song_tower.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for messages, song_ids in loader:
            messages, song_ids = messages.to(device), song_ids.to(device)
            msg_emb = message_tower(messages)
            all_song_ids = torch.arange(song_tower.embedding.num_embeddings).to(device)
            song_embs = song_tower(all_song_ids)
            sims = torch.matmul(msg_emb, song_embs.T)
            topk = torch.topk(sims, k=k, dim=1).indices
            for i in range(messages.size(0)):
                if song_ids[i] in topk[i]:
                    correct += 1
                total += 1
    return correct / total if total > 0 else 0.0


## Evaluation & Analysis

Bagian ini berisi seluruh proses evaluasi performa model, termasuk top-k accuracy, grid search, cross-validation, dan error analysis.


In [123]:
# --- Fungsi Pelatihan dengan Label Smoothing & Advanced Scheduler ---
def smooth_labels(target, smoothing=0.1):
    # Untuk CosineEmbeddingLoss, target: 1 atau -1
    return target * (1 - smoothing) + smoothing * (-target)

def train_recommendation_model(message_tower, song_tower, train_loader, val_loader, config, device, early_stopping_patience=7, save_path='best_model.pth'):
    message_tower.to(device)
    song_tower.to(device)
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"],
        weight_decay=config.get("weight_decay", 0.0)
    )
    if config.get("use_cosine_scheduler", False):
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    else:
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])
    smoothing = config.get("label_smoothing", 0.0)

    def compute_accuracy_and_loss(loader, message_tower, song_tower, device, criterion):
        message_tower.eval()
        song_tower.eval()
        correct = 0
        total = 0
        total_loss = 0
        with torch.no_grad():
            for messages, song_ids in loader:
                messages, song_ids = messages.to(device), song_ids.to(device)
                msg_emb = message_tower(messages)
                all_song_ids = torch.arange(song_tower.embedding.num_embeddings).to(device)
                song_embs = song_tower(all_song_ids)
                sims = torch.matmul(msg_emb, song_embs.T)
                preds = torch.argmax(sims, dim=1)
                correct += (preds == song_ids).sum().item()
                total += messages.size(0)
                pos_emb = song_embs[song_ids]
                loss_pos = criterion(msg_emb, pos_emb, torch.ones(messages.size(0)).to(device) * (1-smoothing) + smoothing * -1)
                neg_emb = torch.roll(pos_emb, shifts=1, dims=0)
                loss_neg = criterion(msg_emb, neg_emb, -torch.ones(messages.size(0)).to(device) * (1-smoothing) + smoothing * 1)
                total_loss += (loss_pos + loss_neg).item()
        avg_loss = total_loss / len(loader) if len(loader) > 0 else 0.0
        acc = correct / total if total > 0 else 0.0
        return acc, avg_loss

    # ...existing code...


In [124]:
# --- 6. Fungsi untuk Rekomendasi (Inference) ---
def build_song_index(song_tower, song_map, device):
    """Proses semua lagu untuk membuat matriks embedding (indeks)."""
    print("\nMembangun indeks embedding untuk semua lagu...")
    song_tower.eval()
    with torch.no_grad():
        # Buat tensor berisi semua ID lagu
        all_song_ids = torch.arange(len(song_map)).to(device)
        # Dapatkan embedding untuk semua lagu sekaligus
        song_index_embeddings = song_tower(all_song_ids)
    print("Indeks berhasil dibangun.")
    return song_index_embeddings

def recommend_songs(message_text, message_tower, song_index, inv_song_map, vocab, config, device, top_k=10):
    """Memberikan top_k rekomendasi lagu untuk sebuah pesan."""
    message_tower.eval()
    
    # 1. Proses pesan input menjadi embedding
    cleaned_text = clean_text(message_text)
    vectorized = [vocab.get(word, vocab['<UNK>']) for word in cleaned_text.split()]
    if len(vectorized) < config["max_len"]:
        vectorized += [vocab['<PAD>']] * (config["max_len"] - len(vectorized))
    else:
        vectorized = vectorized[:config["max_len"]]
        
    query_tensor = torch.tensor(vectorized, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        query_embedding = message_tower(query_tensor)
        
    # 2. Hitung kemiripan (cosine similarity) antara query dan semua lagu di indeks
    similarities = F.cosine_similarity(query_embedding, song_index)
    
    # 3. Dapatkan top_k lagu dengan similarity tertinggi
    top_results = torch.topk(similarities, k=top_k)
    
    # 4. Kembalikan nama lagu dan skornya
    recommendations = []
    for score, song_idx in zip(top_results.values, top_results.indices):
        song_name = inv_song_map[song_idx.item()]
        recommendations.append({
            "song": song_name,
            "similarity_score": score.item()
        })
    return recommendations

# MAIN EXECUTION


In [125]:
# --- Augmentasi Lanjutan: Synonym Replacement & Random Insertion ---
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
import random

def synonym_replacement(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        candidates = [i for i, w in enumerate(new_words) if len(wordnet.synsets(w)) > 0]
        if not candidates:
            break
        idx = random.choice(candidates)
        synonyms = set()
        for syn in wordnet.synsets(new_words[idx]):
            for l in syn.lemmas():
                synonym = l.name().replace('_', ' ').lower()
                if synonym != new_words[idx]:
                    synonyms.add(synonym)
        if synonyms:
            new_words[idx] = random.choice(list(synonyms))
    return new_words

def random_insertion(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        candidates = [w for w in new_words if len(wordnet.synsets(w)) > 0]
        if not candidates:
            break
        word = random.choice(candidates)
        synonyms = set()
        for syn in wordnet.synsets(word):
            for l in syn.lemmas():
                synonym = l.name().replace('_', ' ').lower()
                if synonym != word:
                    synonyms.add(synonym)
        if synonyms:
            insert_word = random.choice(list(synonyms))
            insert_pos = random.randint(0, len(new_words))
            new_words.insert(insert_pos, insert_word)
    return new_words

def advanced_augment_text(text, p_del=0.1, n_swap=1, n_syn=1, n_ins=1, p_aug=0.5):
    words = text.split()
    if len(words) == 0:
        return text
    if random.random() < p_aug:
        words = random_deletion(words, p=p_del)
    if random.random() < p_aug and len(words) > 1:
        words = random_swap(words, n=n_swap)
    if random.random() < p_aug and len(words) > 1:
        words = synonym_replacement(words, n=n_syn)
    if random.random() < p_aug and len(words) > 1:
        words = random_insertion(words, n=n_ins)
    return ' '.join(words)


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\62812\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Augmentasi Lanjutan: Synonym Replacement & Random Insertion

Fungsi-fungsi berikut menambahkan variasi pada pesan training dengan synonym replacement dan random insertion, sehingga model lebih robust terhadap variasi bahasa alami.


---

## 6. Advanced Experiments & Improvement Roadmap

Bagian ini merangkum eksperimen lanjutan dan saran peningkatan performa model yang telah/akan diterapkan, seperti augmentasi lanjutan, oversampling, pretrained embedding, attention, ensemble, hyperparameter search, dan lain-lain.


In [126]:
# --- Pra-pemrosesan Data Asli ---
# Gabungkan song_id ke full_df jika belum ada
if 'song_id' not in full_df.columns:
    full_df = full_df.merge(song_df[['song_name', 'song_id']], on='song_name', how='left')

# Hapus baris dengan nilai kosong di kolom penting
full_df.dropna(subset=['message', 'song_name', 'song_id'], inplace=True)
# Hapus pesan atau nama lagu yang hanya berisi spasi
full_df = full_df[full_df['message'].str.strip() != '']
full_df = full_df[full_df['song_name'].str.strip() != '']
full_df.reset_index(drop=True, inplace=True)

# Filter lagu yang jarang muncul untuk mengurangi noise (hanya di train)
song_counts = full_df['song_id'].value_counts()
frequent_songs = song_counts[song_counts >= CONFIG["min_song_freq"]].index
full_df = full_df[full_df['song_id'].isin(frequent_songs)].reset_index(drop=True)

print(f"Dataset gabungan dimuat dan dibersihkan. Jumlah data setelah filtering: {len(full_df)}")

full_df['cleaned_message'] = full_df['message'].apply(clean_text)

# Split train/validation
from sklearn.model_selection import train_test_split
train_split, val_split = train_test_split(full_df, test_size=0.15, random_state=42, stratify=full_df['song_id'])
print(f"Train: {len(train_split)}, Validation: {len(val_split)}")

# Augmentasi pesan pada train_split
train_split['augmented_message'] = train_split['cleaned_message'].apply(lambda x: augment_text(x, p_del=0.15, n_swap=2, p_aug=0.7))

Dataset gabungan dimuat dan dibersihkan. Jumlah data setelah filtering: 48039
Train: 40833, Validation: 7206


In [127]:
# Terapkan advanced augmentasi pada train_split
train_split['augmented_message'] = train_split['cleaned_message'].apply(
    lambda x: advanced_augment_text(x, p_del=0.15, n_swap=2, n_syn=1, n_ins=1, p_aug=0.7)
)

# --- Oversampling lagu minoritas pada train_split ---
from sklearn.utils import resample
song_counts = train_split['song_id'].value_counts()
min_count = song_counts.min()
max_count = song_counts.max()

# Oversample semua lagu ke jumlah lagu terbanyak
over_sampled = []
for song_id, group in train_split.groupby('song_id'):
    if len(group) < max_count:
        group_oversampled = resample(group, replace=True, n_samples=max_count, random_state=42)
        over_sampled.append(group_oversampled)
    else:
        over_sampled.append(group)
train_split = pd.concat(over_sampled).reset_index(drop=True)
print(f"Setelah oversampling, train_split: {len(train_split)}")


Setelah oversampling, train_split: 4227654


### Augmentasi Lanjutan & Oversampling

Pada tahap ini, dilakukan augmentasi lanjutan pada pesan training menggunakan synonym replacement dan random insertion. Selain itu, dilakukan oversampling pada lagu minoritas agar distribusi label lebih seimbang, sehingga model tidak bias ke lagu mayoritas.


In [128]:
# 2. Build Vocab dan Song Mapping
vocab = build_vocab(train_split['augmented_message'])
all_song_ids = train_split['song_id'].unique().tolist()
song_map = {song_id: i for i, song_id in enumerate(all_song_ids)}
inv_song_map = {i: song_id for song_id, i in song_map.items()}

vocab_size = len(vocab)
num_songs = len(all_song_ids)
print(f"\nUkuran Vocabulary: {vocab_size}, Jumlah Lagu Unik: {num_songs}")


Ukuran Vocabulary: 75370, Jumlah Lagu Unik: 1437


In [129]:
# 3. Create Dataset & DataLoader
X_train = train_split['augmented_message'].values
y_train = train_split['song_id'].values
X_val = val_split['cleaned_message'].values
y_val = val_split['song_id'].values

train_dataset = RecommendationDataset(X_train, y_train, vocab, song_map, CONFIG["max_len"])
val_dataset = RecommendationDataset(X_val, y_val, vocab, song_map, CONFIG["max_len"])
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

In [130]:
# 4. Inisialisasi Model
message_tower = MessageTower(vocab_size, CONFIG)
song_tower = SongTower(num_songs, CONFIG)

In [131]:
# 4. Inisialisasi Model
# Model 1: GRU-only
message_tower_gru = MessageTower(vocab_size, CONFIG, vocab)
song_tower_gru = SongTower(num_songs, CONFIG)

# Model 2: GRU + Attention
CONFIG_ATT = CONFIG.copy()
CONFIG_ATT["use_attention"] = True
message_tower_attn = MessageTower(vocab_size, CONFIG_ATT, vocab)
song_tower_attn = SongTower(num_songs, CONFIG_ATT)


### Inisialisasi Dua Model

Pada tahap ini, dua arsitektur model diinisialisasi: satu model GRU-only dan satu model GRU+Attention. Keduanya akan digunakan untuk eksperimen ensemble dan perbandingan performa.


In [ ]:
# --- Error Analysis pada Validation ---
def error_analysis(val_loader, message_tower, song_tower, inv_song_map, device, n_samples=10):
    message_tower.eval()
    song_tower.eval()
    errors = []
    with torch.no_grad():
        for messages, song_ids in val_loader:
            messages, song_ids = messages.to(device), song_ids.to(device)
            # Pastikan messages di device yang sama dengan model
            messages = messages.to(device)
            msg_emb = message_tower(messages)
            all_song_ids = torch.arange(song_tower.embedding.num_embeddings).to(device)
            song_embs = song_tower(all_song_ids)
            sims = torch.matmul(msg_emb, song_embs.T)
            preds = torch.argmax(sims, dim=1)
            for i in range(messages.size(0)):
                if preds[i] != song_ids[i]:
                    errors.append((messages[i].cpu().numpy(), song_ids[i].item(), preds[i].item()))
    print(f"Total error samples: {len(errors)}")
    for i, (msg_vec, true_id, pred_id) in enumerate(errors[:n_samples]):
        msg_words = [k for k,v in vocab.items() if v in msg_vec and v > 1]
        print(f"Sample {i+1} | Pesan: {' '.join(msg_words)} | True: {inv_song_map[true_id]} | Pred: {inv_song_map[pred_id]}")

# Contoh pemanggilan:
error_analysis(val_loader, message_tower_gru, song_tower_gru, inv_song_map, device)


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

# Error Analysis pada Validation

Analisis error dilakukan untuk memahami kasus-kasus di mana model salah merekomendasikan lagu. Dengan melihat beberapa contoh pesan, label lagu sebenarnya, dan prediksi model, kita dapat mengidentifikasi pola error dan peluang perbaikan lebih lanjut.


In [ ]:
# --- K-Fold Cross-Validation ---
from sklearn.model_selection import StratifiedKFold
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_scores = []
for fold, (train_idx, val_idx) in enumerate(kfold.split(full_df, full_df['song_id'])):
    print(f"\nFold {fold+1}")
    train_fold = full_df.iloc[train_idx].copy()
    val_fold = full_df.iloc[val_idx].copy()
    train_fold['augmented_message'] = train_fold['cleaned_message'].apply(lambda x: advanced_augment_text(x, p_del=0.15, n_swap=2, n_syn=1, n_ins=1, p_aug=0.7))
    vocab_fold = build_vocab(train_fold['augmented_message'])
    song_ids_fold = train_fold['song_id'].unique().tolist()
    song_map_fold = {song_id: i for i, song_id in enumerate(song_ids_fold)}
    train_dataset_fold = RecommendationDataset(train_fold['augmented_message'].values, train_fold['song_id'].values, vocab_fold, song_map_fold, CONFIG["max_len"])
    val_dataset_fold = RecommendationDataset(val_fold['cleaned_message'].values, val_fold['song_id'].values, vocab_fold, song_map_fold, CONFIG["max_len"])
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader_fold = DataLoader(val_dataset_fold, batch_size=CONFIG["batch_size"], shuffle=False)
    mt_fold = MessageTower(len(vocab_fold), CONFIG, vocab_fold)
    st_fold = SongTower(len(song_ids_fold), CONFIG)
    train_recommendation_model(mt_fold, st_fold, train_loader_fold, val_loader_fold, CONFIG, device, early_stopping_patience=3)
    val_acc, _ = train_recommendation_model.compute_accuracy_and_loss(val_loader_fold, mt_fold, st_fold, device, nn.CosineEmbeddingLoss(margin=CONFIG["margin"]))
    val_scores.append(val_acc)
print(f"\nRata-rata Validation Accuracy (5-fold): {np.mean(val_scores):.4f}")


# K-Fold Cross-Validation

K-Fold cross-validation digunakan untuk mengukur generalisasi model pada berbagai pembagian data. Data dibagi menjadi 5 fold, dan model dilatih serta dievaluasi pada setiap fold. Rata-rata akurasi validation dari seluruh fold memberikan estimasi performa yang lebih robust dan mengurangi bias dari satu split saja.


In [ ]:
# --- Evaluasi Top-5 dan Top-10 Accuracy pada Validation ---
top5_acc = top_k_accuracy(val_loader, message_tower_gru, song_tower_gru, device, k=5)
top10_acc = top_k_accuracy(val_loader, message_tower_gru, song_tower_gru, device, k=10)
print(f"Top-5 Accuracy (GRU-only): {top5_acc:.4f}")
print(f"Top-10 Accuracy (GRU-only): {top10_acc:.4f}")

top5_acc_attn = top_k_accuracy(val_loader, message_tower_attn, song_tower_attn, device, k=5)
top10_acc_attn = top_k_accuracy(val_loader, message_tower_attn, song_tower_attn, device, k=10)
print(f"Top-5 Accuracy (GRU+Attention): {top5_acc_attn:.4f}")
print(f"Top-10 Accuracy (GRU+Attention): {top10_acc_attn:.4f}")


# Evaluasi Top-5 dan Top-10 Accuracy pada Validation

Cell ini menjalankan evaluasi top-5 dan top-10 accuracy pada validation set untuk dua model: GRU-only dan GRU+Attention. Hasil ini memberikan gambaran seberapa sering lagu target muncul di 5 atau 10 rekomendasi teratas, yang sangat relevan untuk aplikasi nyata.


In [ ]:
# --- Grid Search Sederhana untuk Hyperparameter Tuning ---
from itertools import product

dropouts = [0.3, 0.5]
learning_rates = [0.0001, 0.0003]
hidden_dims = [64, 128]
best_val_acc = 0
best_config = None
for dropout, lr, hdim in product(dropouts, learning_rates, hidden_dims):
    config_gs = CONFIG.copy()
    config_gs["dropout"] = dropout
    config_gs["learning_rate"] = lr
    config_gs["hidden_dim"] = hdim
    print(f"\nGrid Search: dropout={dropout}, lr={lr}, hidden_dim={hdim}")
    mt = MessageTower(vocab_size, config_gs, vocab)
    st = SongTower(num_songs, config_gs)
    train_recommendation_model(mt, st, train_loader, val_loader, config_gs, device, early_stopping_patience=3)
    # Evaluasi val acc
    val_acc, _ = train_recommendation_model.compute_accuracy_and_loss(val_loader, mt, st, device, nn.CosineEmbeddingLoss(margin=config_gs["margin"]))
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_config = config_gs
print(f"\nBest config: {best_config}, Best val acc: {best_val_acc}")


# Hyperparameter Search (Grid Search)

Grid search digunakan untuk mencari kombinasi hyperparameter terbaik (dropout, learning rate, hidden dim) yang menghasilkan akurasi validation tertinggi. Proses ini penting untuk mengoptimalkan performa model tanpa overfitting.


In [ ]:
# 5. Latih Model
print("Training model GRU-only...")
train_recommendation_model(message_tower_gru, song_tower_gru, train_loader, val_loader, CONFIG, device)

print("\nTraining model GRU+Attention...")
train_recommendation_model(message_tower_attn, song_tower_attn, train_loader, val_loader, CONFIG_ATT, device)

# --- Ensemble: rata-rata skor similarity dari dua model ---
def ensemble_recommend_songs(message_text, towers, song_indices, inv_song_map, vocab, config, device, top_k=10):
    scores = []
    for (message_tower, song_index) in zip(towers, song_indices):
        message_tower.eval()
        cleaned_text = clean_text(message_text)
        vectorized = [vocab.get(word, vocab['<UNK>']) for word in cleaned_text.split()]
        if len(vectorized) < config["max_len"]:
            vectorized += [vocab['<PAD>']] * (config["max_len"] - len(vectorized))
        else:
            vectorized = vectorized[:config["max_len"]]
        query_tensor = torch.tensor(vectorized, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            query_embedding = message_tower(query_tensor)
        similarities = F.cosine_similarity(query_embedding, song_index)
        scores.append(similarities)
    avg_score = torch.stack(scores).mean(dim=0)
    top_results = torch.topk(avg_score, k=top_k)
    recommendations = []
    for score, song_idx in zip(top_results.values, top_results.indices):
        song_name = inv_song_map[song_idx.item()]
        recommendations.append({"song": song_name, "similarity_score": score.item()})
    return recommendations


# Training Dua Model dan Ensemble

Pada bagian ini, dua model dilatih: satu model GRU-only dan satu model GRU+Attention. Setelah training, tersedia fungsi ensemble yang menggabungkan skor similarity dari kedua model untuk menghasilkan rekomendasi yang lebih robust.


In [ ]:
# 6. Bangun Indeks untuk Inference
song_index = build_song_index(song_tower, song_map, device)


Membangun indeks embedding untuk semua lagu...
Indeks berhasil dibangun.


In [ ]:
# 7. Tes Rekomendasi
print("\n--- Tes Rekomendasi ---")
test_message_1 = "kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa"
test_message_2 = "lu gila banget!"

recs_1 = recommend_songs(test_message_1, message_tower, song_index, inv_song_map, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_1}'")
for rec in recs_1:
   print(f"  - {rec['song']} (Score: {rec['similarity_score']:.4f})")
   
recs_2 = recommend_songs(test_message_2, message_tower, song_index, inv_song_map, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_2}'")
for rec in recs_2:
   print(f"  - {rec['song']} (Score: {rec['similarity_score']:.4f})")


--- Tes Rekomendasi ---

Rekomendasi untuk pesan: 'kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa'
  - 0IktbUcnAGrvD03AWnz3Q8 (Score: 0.9316)
  - 5tyGpIXFvaSwfCx2A82rR8 (Score: 0.7540)
  - 2U3jOPfO4wZZFaaWS4Dcj6 (Score: 0.5116)
  - 1GCbc1vpkZA2zhjsSFhmHT (Score: 0.4699)
  - 50x1Ic8CaXkYNvjmxe3WXy (Score: 0.4428)
  - 0bht8SpPHXiUMApKcua4Mz (Score: 0.4206)
  - 4M1wfKr9MlDT2jVgDCMAnA (Score: 0.4078)
  - 4VQH4VluDUOsOuDxccTeyN (Score: 0.4053)
  - 6f4HCP1JLS0ia5UdZ7hgJV (Score: 0.3901)
  - 4mOjjXKDOR1sel6APULUaN (Score: 0.3874)

Rekomendasi untuk pesan: 'lu gila banget!'
  - 0IktbUcnAGrvD03AWnz3Q8 (Score: 0.9584)
  - 5tyGpIXFvaSwfCx2A82rR8 (Score: 0.7290)
  - 1GCbc1vpkZA2zhjsSFhmHT (Score: 0.4504)
  - 50x1Ic8CaXkYNvjmxe3WXy (Score: 0.4383)
  - 4VQH4VluDUOsOuDxccTeyN (Score: 0.4376)
  - 0bht8SpPHXiUMApKcua4Mz (Score: 0.4276)
  - 4M1wfKr9MlDT2jVgDCMAnA (Score: 0.4084)
  - 0Cy7wt6IlRfBPHXXjmZbcP (Score: 0.3985)
  - 6f4HCP1JLS0ia5UdZ7hgJV (Score: 0.3968)
  - 4mOjjXKDOR1sel6AP